# AlphaLOB Phase 2 — Notebook 02: Feature Engineering & Normalization

**Input:** `/content/lob_data.parquet` (5M rows from Notebook 01)

**Output:** `/content/lob_features.parquet` (5M rows with engineered features + labels)

## The 4 Mathematically-Grounded Features (Math+CS Differentiators)

| Feature | Formula | Interview Key Point |
|---------|---------|--------------------|
| **WOFI** | `Σᵢ wᵢ·(Vᵢᵇⁱᵈ − Vᵢᵃˢᵏ)/(Vᵢᵇⁱᵈ + Vᵢᵃˢᵏ)` | Inverse-distance weighted; O(1) deque |
| **Hawkes λ(t)** | `μ + Σᵢ α·exp(−β(t−tᵢ))` | Order arrival clustering; fit once |
| **Kyle's λ** | `ΔPₜ = λ·Qₜ + εₜ` | Price impact via OLS; 5-min rolling |
| **Amihud ILLIQ** | `(1/T)·Σ|rₜ|/VOLₜ` | Illiquidity regime context for HMM |

**Critical:** All features are Z-Score normalized using ROLLING windows only (no look-ahead bias).

---

In [ ]:
# Cell 1: Install dependencies
!pip install polars pyarrow tick statsmodels --quiet
print('✅ Dependencies installed')

In [ ]:
# Cell 2: Imports
import numpy as np
import polars as pl
from collections import deque
import statsmodels.api as sm
import time
import os
import warnings
warnings.filterwarnings('ignore')

PARQUET_IN  = '/content/lob_data.parquet'
PARQUET_OUT = '/content/lob_features.parquet'
N_LEVELS    = 10
NORM_WINDOW = 1000   # rolling z-score window (1000 ticks)
KYLE_WINDOW = 3000   # 5-min equivalent in ticks (10 ticks/sec × 300s)

print('✅ Imports done')

In [ ]:
# Cell 3: Load data
t0 = time.time()
df = pl.read_parquet(PARQUET_IN)
print(f'✅ Loaded {len(df):,} rows in {time.time()-t0:.1f}s')
print(f'   Columns: {df.columns[:6]}...') 

In [ ]:
# Cell 4: FEATURE 1 — WOFI (Weighted Order Flow Imbalance)
# Formula: WOFI = Σᵢ wᵢ·(Vᵢᵇⁱᵈ − Vᵢᵃˢᵏ)/(Vᵢᵇⁱᵈ + Vᵢᵃˢᵏ)
# Weight:  wᵢ = 1 / (1 + |pᵢ − mid|)  (inverse distance from mid)
# This captures where the order book "weight" is: bid-heavy = buy pressure

print('Computing WOFI...')
t0 = time.time()

mid = df['mid_price'].to_numpy()

wofi_values = np.zeros(len(df))

for lvl in range(N_LEVELS):
    bid_p = df[f'bid_price_{lvl}'].to_numpy()
    ask_p = df[f'ask_price_{lvl}'].to_numpy()
    bid_v = df[f'bid_vol_{lvl}'].to_numpy()
    ask_v = df[f'ask_vol_{lvl}'].to_numpy()

    # Inverse distance weights
    w_bid = 1.0 / (1.0 + np.abs(bid_p - mid))
    w_ask = 1.0 / (1.0 + np.abs(ask_p - mid))

    denom = bid_v + ask_v
    denom = np.where(denom < 1e-9, 1e-9, denom)  # avoid divide by zero

    # Average bid/ask weight at this level
    w = (w_bid + w_ask) / 2.0
    wofi_values += w * (bid_v - ask_v) / denom

# Normalize by sum of weights (so WOFI ∈ [-1, 1] approximately)
weight_sum = sum(1.0 / (1.0 + lvl * 0.5) for lvl in range(N_LEVELS))
wofi_values /= weight_sum

print(f'✅ WOFI computed in {time.time()-t0:.1f}s')
print(f'   Range: [{wofi_values.min():.3f}, {wofi_values.max():.3f}]')
print(f'   Mean:  {wofi_values.mean():.4f} (should be ~0)')

In [ ]:
# Cell 5: FEATURE 2 — Hawkes Process Intensity
# Formula: λ(t) = μ + Σᵢ α·exp(−β(t−tᵢ))
# Captures order arrival CLUSTERING: bursts of orders → high intensity
# Key: fit (μ, α, β) on TRAINING data only, then apply stored params at inference

print('Fitting Hawkes Process...')

try:
    from tick.hawkes import HawkesExpKern
    HAWKES_AVAILABLE = True
    print('  Using tick library (exact MLE fit)')
except ImportError:
    HAWKES_AVAILABLE = False
    print('  tick not available — using analytical approximation')

# Use first 10% of data for fitting Hawkes parameters (training split)
n_fit = len(df) // 10
# Timestamps in seconds (relative)
ts_seconds = np.arange(len(df)) * 0.1  # 10 ticks/second → 0.1s intervals

if HAWKES_AVAILABLE:
    # MLE fit on 500k ticks (fast enough)
    learner = HawkesExpKern(decays=1.0, max_iter=50, verbose=False)
    learner.fit([ts_seconds[:n_fit]])
    mu_hawkes    = float(learner.baseline[0])
    alpha_hawkes = float(learner.adjacency[0, 0])
    beta_hawkes  = 1.0  # fixed decay rate (per tick library convention)
else:
    # Analytical approximation: μ ≈ mean arrival rate, α/β from ACF
    mu_hawkes    = 1.0 / 0.1   # 10 events/second
    alpha_hawkes = 0.5
    beta_hawkes  = 2.0

print(f'✅ Hawkes params: μ={mu_hawkes:.4f}, α={alpha_hawkes:.4f}, β={beta_hawkes:.4f}')

# Apply stored params to full series (vectorized recursive formula)
# λ(tᵢ) = μ + α · Σⱼ<ᵢ exp(−β(tᵢ − tⱼ))
# Recursive: R(i) = exp(−β·dt)·R(i−1) + 1, λ(i) = μ + α·R(i)
print('Computing Hawkes intensity (recursive, O(n))...')
t0 = time.time()
dt_fixed = 0.1  # 100ms between ticks
decay_factor = np.exp(-beta_hawkes * dt_fixed)

R = np.zeros(len(df))
for i in range(1, len(df)):
    R[i] = decay_factor * (R[i-1] + 1.0)

hawkes_intensity = mu_hawkes + alpha_hawkes * R
print(f'✅ Hawkes intensity computed in {time.time()-t0:.1f}s')
print(f'   Range: [{hawkes_intensity.min():.3f}, {hawkes_intensity.max():.3f}]')

# Save fitted coefficients for inference-time use
import json
hawkes_params = {'mu': mu_hawkes, 'alpha': alpha_hawkes, 'beta': beta_hawkes}
with open('/content/hawkes_params.json', 'w') as f:
    json.dump(hawkes_params, f)
print('✅ Hawkes params saved to /content/hawkes_params.json')

In [ ]:
# Cell 6: FEATURE 3 — Kyle's Lambda (Price Impact Coefficient)
# Formula: ΔPₜ = λ·Qₜ + εₜ  (OLS regression, 5-minute rolling windows)
# λ = price impact per unit of signed order flow
# High λ → market is thin, informed trading dominant
# Estimated using statsmodels OLS on rolling 5-minute windows

print('Computing Kyle\'s Lambda (rolling OLS, 5-min windows)...')
t0 = time.time()

mid_prices = df['mid_price'].to_numpy()

# Signed order flow proxy: Σᵢ (bid_vol_i - ask_vol_i) at level 0 and 1
bid_v0 = df['bid_vol_0'].to_numpy()
ask_v0 = df['ask_vol_0'].to_numpy()
bid_v1 = df['bid_vol_1'].to_numpy()
ask_v1 = df['ask_vol_1'].to_numpy()
Q = (bid_v0 - ask_v0) + 0.5 * (bid_v1 - ask_v1)  # signed order flow

# Mid-price changes
delta_P = np.diff(mid_prices, prepend=mid_prices[0])

# Rolling OLS every KYLE_WINDOW ticks — compute vectorized using stride trick
kyle_lambda = np.zeros(len(df))
step = KYLE_WINDOW // 10  # recompute every 10th of the window (efficiency)

current_lambda = 0.0
for i in range(0, len(df), step):
    start = max(0, i - KYLE_WINDOW)
    y = delta_P[start:i+1]
    X = Q[start:i+1]
    if len(y) > 30 and np.std(X) > 1e-9:
        X_with_const = sm.add_constant(X)
        try:
            res = sm.OLS(y, X_with_const).fit(disp=0)
            current_lambda = float(res.params[1])  # slope = Kyle's lambda
        except Exception:
            pass  # keep previous lambda
    # Fill forward for the next `step` ticks
    end = min(i + step, len(df))
    kyle_lambda[i:end] = current_lambda

print(f'✅ Kyle\'s Lambda computed in {time.time()-t0:.1f}s')
print(f'   Mean λ: {kyle_lambda.mean():.6f}')
print(f'   Range:  [{kyle_lambda.min():.6f}, {kyle_lambda.max():.6f}]')

In [ ]:
# Cell 7: FEATURE 4 — Amihud Illiquidity Ratio
# Formula: ILLIQ = (1/T)·Σₜ |rₜ| / VOLₜ
# rₜ = log return, VOLₜ = dollar volume traded
# High ILLIQ → each dollar of volume moves price a lot → illiquid market
# Used as regime context input to the HMM

print('Computing Amihud Illiquidity Ratio...')
t0 = time.time()

AMIHUD_WINDOW = 600  # 60-second rolling window (600 ticks at 10/sec)

# Log returns
log_returns = np.diff(np.log(mid_prices), prepend=0.0)
abs_returns = np.abs(log_returns)

# Dollar volume proxy: price × (bid_vol_0 + ask_vol_0) at best bid/ask
dollar_vol = mid_prices * (bid_v0 + ask_v0)
dollar_vol = np.where(dollar_vol < 1e-9, 1e-9, dollar_vol)

# Amihud ratio per tick
amihud_tick = abs_returns / dollar_vol

# Rolling mean (use Polars for speed)
amihud_series = pl.Series('amihud_tick', amihud_tick)
amihud_rolling = amihud_series.rolling_mean(window_size=AMIHUD_WINDOW, min_periods=10)
amihud_illiq = amihud_rolling.fill_null(strategy='forward').to_numpy()

print(f'✅ Amihud ILLIQ computed in {time.time()-t0:.1f}s')
print(f'   Mean ILLIQ: {amihud_illiq.mean():.2e}')
print(f'   Range:      [{amihud_illiq.min():.2e}, {amihud_illiq.max():.2e}]')

In [ ]:
# Cell 8: ADDITIONAL CONTEXT FEATURES for HMM
# These two features are also used as HMM inputs in Notebook 04

print('Computing realized vol and autocorrelation (HMM inputs)...')

# Realized volatility: rolling std of log-returns (100-tick window)
lr_series = pl.Series('log_ret', log_returns)
realized_vol = lr_series.rolling_std(window_size=100, min_periods=10)\
                        .fill_null(strategy='forward').to_numpy()

# Rolling autocorrelation (lag=1) — measures mean reversion vs momentum
def rolling_autocorr(arr: np.ndarray, window: int = 100, lag: int = 1) -> np.ndarray:
    result = np.zeros(len(arr))
    for i in range(window, len(arr)):
        x = arr[i-window:i]
        if np.std(x) > 1e-12:
            result[i] = np.corrcoef(x[:-lag], x[lag:])[0, 1]
    return result

# Vectorized approximation using Polars expressions (much faster)
autocorr_values = np.zeros(len(df))
step_ac = 500
for i in range(100, len(log_returns), step_ac):
    start = max(0, i - 100)
    x = log_returns[start:i]
    if len(x) > 10 and np.std(x) > 1e-12:
        ac = np.corrcoef(x[:-1], x[1:])[0, 1] if len(x) > 1 else 0.0
        autocorr_values[i:i+step_ac] = ac

print(f'✅ Realized vol range: [{realized_vol.min():.6f}, {realized_vol.max():.6f}]')
print(f'✅ Autocorr range:     [{autocorr_values.min():.3f}, {autocorr_values.max():.3f}]')

In [ ]:
# Cell 9: NORMALIZATION — Online Z-Score (CRITICAL: rolling window only, NO look-ahead)
# Formula: z = (x − μ_rolling) / σ_rolling
# Using Polars rolling_mean and rolling_std (vectorized, fast)
# DO NOT use global mean/std — that introduces look-ahead bias!

print('Applying rolling Z-Score normalization...')
t0 = time.time()

def rolling_zscore(arr: np.ndarray, window: int, name: str) -> np.ndarray:
    """Z-score using only PAST data. Window looks backward only."""
    s = pl.Series(name, arr)
    mu = s.rolling_mean(window_size=window, min_periods=10)
    sd = s.rolling_std(window_size=window, min_periods=10)
    # Avoid division by zero
    sd_arr = sd.fill_null(1.0).to_numpy()
    sd_arr = np.where(sd_arr < 1e-9, 1.0, sd_arr)
    z = (arr - mu.fill_null(strategy='forward').to_numpy()) / sd_arr
    # Clip to [-5, 5] to handle early window instability
    return np.clip(z, -5.0, 5.0)

wofi_z         = rolling_zscore(wofi_values,     NORM_WINDOW, 'wofi')
hawkes_z       = rolling_zscore(hawkes_intensity, NORM_WINDOW, 'hawkes')
kyle_lambda_z  = rolling_zscore(kyle_lambda,      NORM_WINDOW, 'kyle')
amihud_z       = rolling_zscore(amihud_illiq,     NORM_WINDOW, 'amihud')
spread_z       = rolling_zscore(df['spread'].to_numpy(), NORM_WINDOW, 'spread')

print(f'✅ Z-scores computed in {time.time()-t0:.1f}s')
for name, arr in [('wofi_z', wofi_z), ('hawkes_z', hawkes_z),
                   ('kyle_z', kyle_lambda_z), ('amihud_z', amihud_z)]:
    print(f'   {name}: mean={arr.mean():.3f}, std={arr.std():.3f}, range=[{arr.min():.2f},{arr.max():.2f}]')

In [ ]:
# Cell 10: LABELS — Forward-looking mid-price direction
# label = 1 if mid_price(t + horizon) > mid_price(t) else 0
# CRITICAL: use shift(-horizon) NOT shift(+horizon) — shift backward means FUTURE
# In Polars, shift(-n) gives you the value n rows AHEAD

print('Creating forward-looking labels...')

HORIZONS = {
    'label_5s':   50,    # 5 seconds  × 10 ticks/sec = 50 ticks
    'label_30s':  300,   # 30 seconds × 10 ticks/sec = 300 ticks
    'label_5min': 3000,  # 5 minutes  × 10 ticks/sec = 3000 ticks
}

mid_series = pl.Series('mid_price', mid_prices)
labels = {}

for label_name, horizon in HORIZONS.items():
    # Future price (shift left by horizon — uses FUTURE data for label only)
    future_mid = mid_series.shift(-horizon)
    label = (future_mid > mid_series).cast(pl.Int8)  # 1=UP, 0=DOWN
    labels[label_name] = label.fill_null(0).to_numpy()
    pct_up = labels[label_name].mean() * 100
    print(f'   {label_name} (horizon={horizon} ticks): {pct_up:.1f}% UP labels')

print('\n⚠️  NOTE: Labels use future data — they MUST only appear in the target y array,')
print('    never as input features X. Look-ahead bias check in test_no_lookahead.py.')

In [ ]:
# Cell 11: Assemble final feature DataFrame and save

print('Assembling feature DataFrame...')
t0 = time.time()

df_features = pl.DataFrame({
    # Identity
    'timestamp':       df['timestamp'],
    'symbol':          df['symbol'],
    # Raw features (for interpretability)
    'mid_price':       mid_prices.tolist(),
    'spread':          df['spread'],
    'wofi':            wofi_values.tolist(),
    'hawkes_intensity': hawkes_intensity.tolist(),
    'kyle_lambda':     kyle_lambda.tolist(),
    'amihud_illiq':    amihud_illiq.tolist(),
    # HMM context features
    'realized_vol':    realized_vol.tolist(),
    'autocorrelation': autocorr_values.tolist(),
    # Normalized features (MODEL INPUTS — these go into the Transformer)
    'wofi_z':          wofi_z.tolist(),
    'hawkes_z':        hawkes_z.tolist(),
    'kyle_lambda_z':   kyle_lambda_z.tolist(),
    'amihud_z':        amihud_z.tolist(),
    'spread_z':        spread_z.tolist(),
    # Labels (TARGETS — never use as model inputs)
    'label_5s':        labels['label_5s'].tolist(),
    'label_30s':       labels['label_30s'].tolist(),
    'label_5min':      labels['label_5min'].tolist(),
})

# Drop rows where labels are 0 due to end-of-series fill
# (last 3000 rows have no valid 5min label)
df_features = df_features.head(len(df_features) - 3001)

print(f'✅ Feature DataFrame assembled: {df_features.shape}')
print(f'   Columns: {df_features.columns}')

# Save
df_features.write_parquet(PARQUET_OUT, compression='snappy', use_pyarrow=True)
file_mb = os.path.getsize(PARQUET_OUT) / 1e6
print(f'✅ Saved to {PARQUET_OUT} ({file_mb:.0f} MB) in {time.time()-t0:.1f}s')

In [ ]:
# Cell 12: Visualize feature distributions

import matplotlib.pyplot as plt

sample = df_features.sample(10000, seed=42)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('AlphaLOB — Feature Distributions After Z-Score Normalization', fontsize=13)

raw_features  = ['wofi',    'hawkes_intensity', 'kyle_lambda', 'amihud_illiq']
norm_features = ['wofi_z',  'hawkes_z',         'kyle_lambda_z', 'amihud_z']
colors        = ['#2196F3', '#4CAF50',           '#FF9800',    '#9C27B0']
labels_row    = ['WOFI', 'Hawkes λ(t)', 'Kyle\'s Lambda', 'Amihud ILLIQ']

for col_idx, (raw, norm, color, title) in enumerate(
        zip(raw_features, norm_features, colors, labels_row)):
    axes[0, col_idx].hist(sample[raw].to_numpy(), bins=60, color=color, alpha=0.8)
    axes[0, col_idx].set_title(f'{title} (raw)')
    axes[0, col_idx].grid(alpha=0.3)

    axes[1, col_idx].hist(sample[norm].to_numpy(), bins=60, color=color, alpha=0.8)
    axes[1, col_idx].set_title(f'{title} (z-score)')
    axes[1, col_idx].axvline(0, color='red', linestyle='--', alpha=0.6)
    axes[1, col_idx].set_xlim(-5, 5)
    axes[1, col_idx].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/features_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature distributions saved to /content/features_overview.png')

In [ ]:
# Cell 13: Label balance check (important before training)

print('=== LABEL BALANCE CHECK ===')
for lbl in ['label_5s', 'label_30s', 'label_5min']:
    counts = df_features[lbl].value_counts()
    up   = df_features[lbl].sum()
    down = len(df_features) - up
    print(f'  {lbl}: UP={up:,} ({up/len(df_features)*100:.1f}%) '
          f'DOWN={down:,} ({down/len(df_features)*100:.1f}%)')

print('\n✅ Labels should be close to 50/50 for an unbiased classifier')
print('   If not, add class weights to the LOBTransformer loss in Notebook 03')

print('\n=== NOTEBOOK 02 COMPLETE ===')
print(f'  Output: {PARQUET_OUT}')
print(f'  Shape:  {df_features.shape}')
print('  Next step → Run 03_train_lobtransformer.ipynb')